# =========================================================
# Component: Memory
# State management of tools / agent
# ========================================================

In [1]:
# **************************
# (example 1) simple history
# **************************

import os
from langchain_openai import ChatOpenAI
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import AIMessage, HumanMessage

llm = ChatOpenAI() # add the api_key parameter here

history = ChatMessageHistory()

C:\Users\sriraman.rajagopalan\AppData\Local\Temp\ipykernel_31416\390659407.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [ ]:
history.add_user_message("hello") # user's first message
history.add_ai_message("Greetings. How are you ?") # AI's reply

history.add_user_message("i am fine. today i want to talk about some interesting topics") # user message
history.add_ai_message("Nice. Looking forward to this.") # AI reply

print(history)

In [ ]:
history.messages

In [ ]:
# *******************************************
# (example 2) : Simulation of a real-life chat
# ********************************************
history = ChatMessageHistory()

# Start the chat loop
msg = ''' Hi. I am your AI Doctor. How can i help you?
        To end this conversation, type exit/bye/quit/close
        '''
flag = True
exit_val = ["exit","quit","bye","close"]

while flag:
    user_input = input("You: ")
    if any(ev for ev in exit_val if ev in user_input):
        flag = False

    # Add user message to history
    history.add_user_message(user_input)

    # Generate AI response using chat history
    ai_response = llm.invoke(history.messages)

    # Display and store AI response
    print(f"User: {user_input}")
    print(f"AI: {ai_response.content}")
    history.add_ai_message(ai_response.content)

In [ ]:
# total messages
print(f"There are {len(history.messages)} messages")

In [ ]:
# display the full chat history
print("\n--- Chat History ---")
for msg in history.messages:
    role = "User" if isinstance(msg, HumanMessage) else "AI"
    print(f"{role}: {msg.content}")

In [ ]:
# Answer for a particular Question number
    
q = 2

print(f"Question Number: {q}")
print(f"Q:" + history.messages[q].content)
print()
print(f"A:" + history.messages[q+1].content)


# A simple langchain example to simulate a Manual entity memory.
# Attached dataset (CSV) is the data source.
# All writes in MySQL table

Entity Memory in LangChain is a specialized memory module that extracts, stores, and updates facts about specific entities (people, places, organizations, objects) mentioned during a conversation.

In [ ]:
import os
import pandas as pd
import mysql.connector
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# root = os.getcwd()
# root = os.path.dirname(root)
# print(root)
# csv_path = root + r"\dataset\household_energy_requirement.csv"
# print(csv_path)

csv_path = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\1\code\dataset\household_energy_requirement.csv"

In [ ]:
energy_data = pd.read_csv(csv_path)
print(energy_data.head())

In [28]:
# Connect to the MySQL database
conn = mysql.connector.connect(host="localhost", user="root", password="root", database="ey")
print(conn)

In [ ]:
energy_data.columns

In [20]:
# The entity data is the memory data that comes from the CSV

entities = [ "entid", "house_type", "num_occupants", "has_ac", "num_ac_units", "daily_energy_consumption_kWh", 
            "monthly_energy_consumption_kWh","estimated_peak_load_kW"]

entid = "ENT100"
ent_data = energy_data[entities][energy_data.entid == entid]

# Convert the output into a JSON format
ent_data = ent_data.iloc[0].to_dict()
print(ent_data)

{'entid': 'ENT100', 'house_type': 'Apartment', 'num_occupants': 1, 'has_ac': 0, 'num_ac_units': 0, 'daily_energy_consumption_kWh': 16.04, 'monthly_energy_consumption_kWh': 481.1, 'estimated_peak_load_kW': 2.0}


In [21]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = PromptTemplate(input_variables=["ent_data", "question"],
    template="""
You are an energy advisor.

Use the following stored entity memory:

{ent_data}

Question:
{question}

Answer in simple terms.
"""
)

chain = prompt | llm | StrOutputParser()

In [22]:
query = "Is this household likely to have high energy consumption? Explain why"

response = chain.invoke({ "ent_data": ent_data, "question": query})
print(response)

This household is not likely to have high energy consumption. Here’s why:

1. **Type of Home**: It's an apartment, which usually means smaller space and lower energy needs compared to larger homes.

2. **Number of Occupants**: There is only 1 person living there, so the energy use is generally lower.

3. **Air Conditioning**: The household does not have air conditioning units, which can significantly increase energy consumption, especially in hot weather.

4. **Daily and Monthly Consumption**: The daily energy consumption is about 16.04 kWh, which totals around 481.1 kWh per month. This is a moderate amount for a single occupant in an apartment.

5. **Peak Load**: The estimated peak load is 2.0 kW, which is relatively low, indicating that the maximum energy demand at any time is not very high.

Overall, considering these factors, this household's energy consumption is on the lower side.


In [23]:
# Insert the response in the MySQL

def ExecuteQuery(action, query, values=None):
    ret = {"status": '', "message": "", "record": ""}
    act_msg = ''
    CURSOR = conn.cursor(dictionary=True)

    try:
        if action == "I":
            act_msg = "Inserted"
        elif action == "U":
            act_msg = "Updated"
        elif action == "D":
            act_msg = "Deleted"
        elif action == "S":
            act_msg = "Retrieved"

        if action in ["I", "U", "D"]:
            CURSOR.execute(query, values)
            conn.commit()
            ret["status"] = "SUCCESS"
            ret["message"] = f"{CURSOR.rowcount} Record(s) {act_msg}"
            ret["record"] = ''
        elif action == "S":
            if values is not None:
                CURSOR.execute(query, values)
            else:
                CURSOR.execute(query)

            data = CURSOR.fetchall()
            ret["status"] = "SUCCESS"
            ret["message"] = f"{len(data)} Record(s) {act_msg} "
            ret["record"] = pd.DataFrame(data)

    except Exception as e:
        ret["status"] = "EXCEPTION"
        ret["message"] = str(e)
        ret["record"] = ''
        CURSOR.close()

    finally:
        CURSOR.close()

    return (ret)


In [24]:
ent_data

{'entid': 'ENT100',
 'house_type': 'Apartment',
 'num_occupants': 1,
 'has_ac': 0,
 'num_ac_units': 0,
 'daily_energy_consumption_kWh': 16.04,
 'monthly_energy_consumption_kWh': 481.1,
 'estimated_peak_load_kW': 2.0}

In [25]:
entity_values = ''

for d in ent_data.values():
    entity_values += str(d) + "~"

print(entity_values)

query = '''INSERT INTO entity_memory(entity_id, entity_name,memory_value) VALUES(%s,%s,%s) '''
values = (entid,entity_values,response)

ENT100~Apartment~1~0~0~16.04~481.1~2.0~


In [29]:
status = ExecuteQuery(action="I",query=query,values=values)

In [30]:
print(status)

{'status': 'SUCCESS', 'message': '1 Record(s) Inserted', 'record': ''}


# LangChain Entity Memory simulation Example

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationEntityMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# initialize memory
memory = ConversationEntityMemory(llm=llm)

In [ ]:
# define prompt template

prompt = PromptTemplate(
    input_variables=["input", "entities"],
    template="""
You are an energy advisor.

Here is what you know about entities so far:
{entities}

User query:
{input}

Answer in simple terms.
"""
)

In [ ]:
# build the Chain
chain = LLMChain(llm=llm, prompt=prompt, memory=memory)

In [ ]:
initial_fact = (
    "ENT100 is a household with 4 occupants, has AC, "
    "2 AC units, daily energy consumption of 18 kWh, "
    "monthly consumption of 540 kWh, and an estimated peak load of 4.2 kW."
)

chain.run(initial_fact)

In [ ]:
query = "Is this household likely to have high energy consumption? Explain why"
response = chain.run(query)
print(response)

# =========================================================
# Component: Vector Store
# Create embeddings on input data and perform Q/A
# ========================================================


In [31]:
#  pip install faiss-cpu tiktoken

import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [32]:
f1 = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\1\code\dataset\ai.txt"
f2 = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\1\code\dataset\indian_classical_music.pdf"

In [33]:
# Step 1: Load text from file
loader = TextLoader(f1)
documents = loader.load()

# Step 2: Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=20)
docs = text_splitter.split_documents(documents)
print(f"There are {len(docs)} chunks")

There are 14 chunks


In [34]:
# Step 3: print the chunks and its length

chunks = [doc.page_content for doc in docs]

for ndx, c in enumerate(chunks,start=1):
    print(f"Chunk = {ndx}, Text = {c}")

Chunk = 1, Text = Artificial intelligence (AI), in its broadest sense, is intelligence exhibited by machines, particularly computer systems.
Chunk = 2, Text = It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment
Chunk = 3, Text = and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1] Such machines may be called AIs.
Chunk = 4, Text = Some high-profile applications of AI include advanced web search engines (e.g., Google Search); 
recommendation systems (used by YouTube, Amazon, and Netflix); interacting via human speech (e.g., Google Assistant, Siri, and Alexa);
Chunk = 5, Text = autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., ChatGPT, and AI art); and superhuman play and 
analysis in strategy games (e.g., chess and Go). However, many AI applications are not
Chunk = 6, Text = perceived as AI: "A lot of cutting edge AI ha

In [35]:
# Step 4: Convert to vector store using OpenAI
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_texts(chunks,embedding=embeddings)

In [36]:
# Step 5: Perform similarity search
query = "Give 3 examples of real world AI"
results = vectorstore.similarity_search(query, k=2)

In [37]:
# Step 6: Display results
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n{doc.page_content}")



--- Result 1 ---
Some high-profile applications of AI include advanced web search engines (e.g., Google Search); 
recommendation systems (used by YouTube, Amazon, and Netflix); interacting via human speech (e.g., Google Assistant, Siri, and Alexa);

--- Result 2 ---
autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., ChatGPT, and AI art); and superhuman play and 
analysis in strategy games (e.g., chess and Go). However, many AI applications are not


In [ ]:
# Example 2

query = "how was AI founded"
results = vectorstore.similarity_search(query, k=1)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n{doc.page_content}")

# Working with LangChain agents

# Implements the following:

# 1) Dynamic routing and decision making 
# 2) Multi-step reasoning and iterative execution 
# 3) Human-in-the-loop validation 
# 4) Guardrails and execution constraints

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# --------------------------------------------------
# Tools: Python functions exposed as LangChain tools
# --------------------------------------------------

@tool
def calculate_gst(invoice_amount: float, gst_rate: float) -> dict:
    """Calculate GST amount and total invoice value."""
    gst_amount = invoice_amount * gst_rate / 100
    total_amount = invoice_amount + gst_amount

    return {
        "invoice_amount": invoice_amount,
        "gst_rate": gst_rate,
        "gst_amount": gst_amount,
        "total_amount": total_amount
    }


@tool
def calculate_tds(invoice_amount: float, tds_rate: float) -> dict:
    """Calculate TDS deduction and net payable amount."""
    tds_amount = invoice_amount * tds_rate / 100
    net_payable = invoice_amount - tds_amount

    return {
        "invoice_amount": invoice_amount,
        "tds_rate": tds_rate,
        "tds_amount": tds_amount,
        "net_payable": net_payable
    }


@tool
def check_invoice_limit(total_amount: float) -> str:
    """Check whether invoice amount is within payment approval limit."""
    if total_amount > 500000:
        return "REQUIRES SENIOR APPROVAL"
    return "WITHIN APPROVAL LIMIT"

In [ ]:
# Register the Tools
tools = [calculate_gst, calculate_tds, check_invoice_limit]

tool_registry = {tool.name: tool for tool in tools}

tool_registry

In [ ]:
# --------------------------------------------------
# Guardrail
# --------------------------------------------------

def guardrail_check(user_prompt: str):
    blocked_words = ["delete", "drop", "bypass approval", "ignore tax"]

    for word in blocked_words:
        if word in user_prompt.lower():
            return False

    return True

In [ ]:
# --------------------------------------------------
# Tool selection prompt
# --------------------------------------------------

sys_prompt = """
You are a tax and finance routing agent.

Select the correct tool from the list:

1. calculate_gst
Use this for GST calculation, tax invoice value, total invoice value.

2. calculate_tds
Use this for TDS deduction, withholding tax, net payable amount.

3. check_invoice_limit
Use this for invoice approval limit or payment approval check.

Return only the tool name.
"""

tool_selection_prompt = ChatPromptTemplate.from_messages([ ("system", sys_prompt), ("human", "{user_prompt}") ])

tool_selection_chain = tool_selection_prompt | llm | StrOutputParser()

In [ ]:
# --------------------------------------------------
# Final explanation prompt
# --------------------------------------------------

final_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a finance analyst.
Explain the result clearly.
Mention the selected tool, calculation result, and next action.
"""),
    ("human", """
User request:
{user_prompt}

Selected tool:
{selected_tool}

Tool result:
{tool_result}
""")
])

final_chain = final_prompt | llm | StrOutputParser()


In [ ]:
# --------------------------------------------------
# Main execution function
# --------------------------------------------------

def run_finance_agent(user_prompt: str, tool_args: dict):

    # 1. Guardrail
    if not guardrail_check(user_prompt):
        return "Blocked by guardrail. Unsafe finance request."

    # 2. Dynamic routing
    selected_tool = tool_selection_chain.invoke({"user_prompt": user_prompt}).strip()

    if selected_tool not in tool_registry:
        return "No Tool Selected"

    # 3. Human-in-the-loop validation
    print("\nSelected Tool:", selected_tool)
    print("Arguments:", tool_args)

    approval = input("Approve tool execution? yes/no: ")

    if approval.lower() != "yes":
        return "Execution stopped by human reviewer."

    # 4. Tool execution
    tool_result = tool_registry[selected_tool].invoke(tool_args)

    # 5. Final response generation
    final_answer = final_chain.invoke({"user_prompt": user_prompt,  "selected_tool": selected_tool,  "tool_result": tool_result })
    return final_answer


In [ ]:
# --------------------------------------------------
# Example 1: GST calculation
# --------------------------------------------------

response = run_finance_agent(
    user_prompt="Calculate GST for vendor invoice amount 250000 with GST rate 18%",
    tool_args={ "invoice_amount": 250000, "gst_rate": 18 }
)

In [ ]:
print(response)